In [ ]:
import keras
import tensorflow as tf 
from preprocessing import train_ds, valid_ds, dice_metric, dice_metric_cup, dice_metric_disc, combined_loss, STEPS_PER_EPOCH, VALID_STEPS

tf.random.set_seed(0)

he_init=keras.initializers.HeNormal()

elu_act=keras.activations.elu

l2_reg=keras.regularizers.l2(1e-4)

inputs=keras.layers.Input(shape=(288,288,3))

# Encoder which downscales the image
x=keras.layers.Conv2D(filters=32, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(inputs)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=32, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
block_1_output=x
x=keras.layers.MaxPool2D((2,2))(x)

skip_layer_1=keras.layers.Conv2D(filters=64, kernel_size=(1,1), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.Conv2D(filters=64, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=64, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.add([x,skip_layer_1])
block_2_output=x
x=keras.layers.MaxPool2D((2,2))(x)

skip_layer_2=keras.layers.Conv2D(filters=128, kernel_size=(1,1), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.Conv2D(filters=128, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=128, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.add([x,skip_layer_2])
block_3_output=x
x=keras.layers.MaxPool2D((2,2))(x)

skip_layer_3=keras.layers.Conv2D(filters=256, kernel_size=(1,1), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.Conv2D(filters=256, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=256, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.add([x,skip_layer_3])
block_4_output=x
x=keras.layers.MaxPool2D((2,2))(x)

x=keras.layers.Conv2D(512, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Dropout(0.1)(x)

x=keras.layers.Conv2D(512, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Dropout(0.1)(x)

bottleneck=x

# Decoder which upscales the images

x=keras.layers.Conv2DTranspose(filters=256,kernel_size=(2,2),strides=2, padding="same", kernel_initializer=he_init, kernel_regularizer=l2_reg)(bottleneck)
x=keras.layers.Concatenate()([x,block_4_output])
x=keras.layers.Dropout(0.2)(x)

block_1_decoder_output=keras.layers.Conv2D(256, kernel_size=(1,1), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)

x=keras.layers.Conv2D(filters=256, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=256, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.add([x, block_1_decoder_output])

x=keras.layers.Conv2DTranspose(filters=128,kernel_size=(2,2),strides=2, padding="same", kernel_initializer=he_init, kernel_regularizer=l2_reg)(x)
x=keras.layers.Concatenate()([x,block_3_output])
x=keras.layers.Dropout(0.2)(x)

block_2_decoder_output=keras.layers.Conv2D(128, kernel_size=(1,1), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)

x=keras.layers.Conv2D(filters=128, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=128, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.add([x, block_2_decoder_output])

x=keras.layers.Conv2DTranspose(filters=64,kernel_size=(2,2),strides=2, padding="same", kernel_initializer=he_init, kernel_regularizer=l2_reg)(x)
x=keras.layers.Concatenate()([x,block_2_output])
x=keras.layers.Dropout(0.2)(x)

block_3_decoder_output=keras.layers.Conv2D(64, kernel_size=(1,1), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)

x=keras.layers.Conv2D(filters=64, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=64, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.add([x, block_3_decoder_output])

x=keras.layers.Conv2DTranspose(filters=32,kernel_size=(2,2),strides=2, padding="same", kernel_initializer=he_init, kernel_regularizer=l2_reg)(x)
x=keras.layers.Concatenate()([x,block_1_output])
x=keras.layers.Dropout(0.2)(x)

block_4_decoder_output=keras.layers.Conv2D(32, kernel_size=(1,1), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)

x=keras.layers.Conv2D(filters=32, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=32, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.add([x, block_4_decoder_output])

outputs=keras.layers.Conv2D(3, kernel_size=(1,1), activation='softmax')(x)

model=keras.Model(inputs, outputs)

adamW=keras.optimizers.AdamW(3e-4, clipnorm=1.0, weight_decay=1e-4)

sgd=keras.optimizers.SGD(1e-3,momentum=0.95, nesterov=True)

model.compile(
    optimizer=adamW,
    loss=combined_loss,
    metrics=[dice_metric, dice_metric_disc, dice_metric_cup]
)

earlyStop_cb=keras.callbacks.EarlyStopping(
    monitor='val_dice_metric',
    patience=10,
    verbose=1,
    restore_best_weights=True,
    mode='max'
)

lrPlateau_cb=keras.callbacks.ReduceLROnPlateau(
    monitor='val_dice_metric',
    patience=3,
    factor=0.5,
    verbose=1,
    min_lr=5e-7,
    mode='max'
)

history=model.fit(
    train_ds,
    batch_size=32,
    epochs=40,
    callbacks=[earlyStop_cb, lrPlateau_cb],
    validation_data=valid_ds,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_steps=VALID_STEPS
)

model.save("Saved Models/Seg_ResidualUNetStyle_adamw_lr3e-4_combinedloss_dicemetric_perclassdicemetricandloss.keras")                                       

2026-04-22 00:48:28.732473: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-04-22 00:48:28.732552: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-04-22 00:48:28.732557: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-04-22 00:48:28.732577: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-22 00:48:28.732590: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Epoch 1/40


2026-04-22 00:48:31.935980: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


50/50 ━━━━━━━━━━━━━━━━━━━━ 463s 9s/step - dice_metric: 0.8806 - dice_metric_cup: 0.4571 - dice_metric_disc: 0.9223 - loss: 9.2336 - val_dice_metric: 0.8588 - val_dice_metric_cup: 4.6129e-04 - val_dice_metric_disc: 0.9243 - val_loss: 17.5926 - learning_rate: 3.0000e-04
Epoch 2/40
50/50 ━━━━━━━━━━━━━━━━━━━━ 244s 5s/step - dice_metric: 0.9919 - dice_metric_cup: 0.7610 - dice_metric_disc: 0.9973 - loss: 2.0623 - val_dice_metric: 0.9660 - val_dice_metric_cup: 5.1269e-10 - val_dice_metric_disc: 0.9845 - val_loss: 5.6299 - learning_rate: 3.0000e-04
Epoch 3/40
50/50 ━━━━━━━━━━━━━━━━━━━━ 199s 4s/step - dice_metric: 0.9922 - dice_metric_cup: 0.7990 - dice_metric_disc: 0.9971 - loss: 1.8642 - val_dice_metric: 0.9837 - val_dice_metric_cup: 1.2947e-09 - val_dice_metric_disc: 0.9918 - val_loss: 5.2996 - learning_rate: 3.0000e-04
Epoch 4/40
50/50 ━━━━━━━━━━━━━━━━━━━━ 216s 4s/step - dice_metric: 0.9925 - dice_metric_cup: 0.8270 - dice_metric_disc: 0.9970 - loss: 1.7232 - val_dice_metric: 0.9838 - val_